## Hybrid Recommender System

The goal of this experiment is to investigate whether combining two collaborative filtering approaches can improve recommendation performance on the MovieLens 100K dataset.

The hybrid recommender combines:

- **ItemKNN**, a neighborhood-based collaborative filtering method that recommends items based on item-item similarity.
- **SLIMElastic**, a sparse linear recommendation model that learns relationships between items using Elastic Net regularization.

The motivation for combining these models is that they capture item relationships in different ways. ItemKNN relies directly on similarity between items, while SLIMElastic learns sparse item-item coefficients from user interaction data. Combining their predictions may therefore provide complementary information and improve recommendation quality.

## Hybrid Method

The hybrid recommender combines the prediction scores produced by ItemKNN and SLIMElastic. Instead of manually assigning the contribution of each recommender, the weights are learned using a regression model.

For each user-item pair, the normalized prediction scores of ItemKNN and SLIMElastic are used as input features:

$$
X_{ui} =
\begin{bmatrix}
S_{\text{ItemKNN}}(u,i) &
S_{\text{SLIM}}(u,i)
\end{bmatrix}
$$

The target variable represents whether the item is relevant to the user:

$$
y_{ui} =
\begin{cases}
1 & \text{if item } i \text{ is relevant to user } u \\
0 & \text{otherwise}
\end{cases}
$$

A linear regression model is then fitted:

$$
\hat{y}_{ui}
=
\beta_0
+
\beta_1 S_{\text{ItemKNN}}(u,i)
+
\beta_2 S_{\text{SLIM}}(u,i)
$$

where $\beta_1$ and $\beta_2$ are learned from the validation data. The resulting predicted value $\hat{y}_{ui}$ is used as the final hybrid ranking score.

This approach allows the contribution of each individual recommender to be learned from data rather than manually specified.

### Evaluation Methodology

The validation set was used to learn the hybrid coefficients. The normalized
prediction scores from ItemKNN and SLIMElastic were used as input features
for a linear regression model.

For each user, relevant validation items were assigned a target value of 1. For each positive interaction, 10 unobserved items that were neither present in the user's training history nor among their validation positives were sampled and assigned a target value of 0.

The learned regression model is:

$$
\hat{y}_{ui}
=
\beta_0
+
\beta_1 S_{\mathrm{ItemKNN}}(u,i)
+
\beta_2 S_{\mathrm{SLIM}}(u,i)
$$

The learned coefficients determine the contribution of each recommender to
the final hybrid score. The test set was kept separate from the
weight-learning process and was used only for final evaluation.

Performance is reported using Recall@10, MRR@10, NDCG@10, Hit@10,
and Precision@10.

In [1]:
import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LinearRegression

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, init_seed


TOP_K = 10
SEED = 42

## Base Recommenders

The best ItemKNN and SLIMElastic configurations obtained during individual-model
tuning are used as the components of the hybrid. Both models are constructed
from the same training split so that their user and item indices are aligned.

In [2]:
item_config = Config(
    model="ItemKNN",
    dataset="ml-100k",
    config_file_list=[
        "../recbole/config/ItemKNN/ml-100k.yaml"
    ],
)

slim_config = Config(
    model="SLIMElastic",
    dataset="ml-100k",
    config_file_list=[
        "../recbole/config/SLIMElastic/ml-100k.yaml"
    ],
)

init_seed(
    item_config["seed"],
    item_config["reproducibility"]
)

dataset = create_dataset(item_config)

train_data, valid_data, test_data = data_preparation(
    item_config,
    dataset
)

item_model = get_model("ItemKNN")(
    item_config,
    train_data.dataset
).to(item_config["device"])

slim_model = get_model("SLIMElastic")(
    slim_config,
    train_data.dataset
).to(slim_config["device"])

item_model.eval()
slim_model.eval()

SLIMElastic()

In [43]:
print("ItemKNN:", item_model.k, item_model.shrink, item_model.method)
print("SLIM:", slim_config["alpha"], slim_config["l1_ratio"])

ItemKNN: 100 100 item
SLIM: 0.1 0.1


In [44]:
def normalize(scores):
    minimum = scores.min(dim=1, keepdim=True).values
    maximum = scores.max(dim=1, keepdim=True).values

    return (scores - minimum) / (
        maximum - minimum + 1e-12
    )


def get_scores(interaction):
    item_scores = item_model.full_sort_predict(
        interaction
    ).view(len(interaction), -1)

    slim_scores = slim_model.full_sort_predict(
        interaction
    ).view(len(interaction), -1)

    return normalize(item_scores), normalize(slim_scores)

In [45]:
def metrics_at_k(recommended, relevant, k=10):
    relevant = set(relevant)
    recommended = recommended[:k]

    hits = np.array([
        item in relevant
        for item in recommended
    ])

    precision = hits.sum() / k
    recall = hits.sum() / len(relevant) if relevant else 0
    hit = float(hits.any())

    positions = np.where(hits)[0]
    mrr = 1 / (positions[0] + 1) if len(positions) else 0

    discounts = 1 / np.log2(np.arange(2, k + 2))
    dcg = (hits * discounts).sum()

    ideal_hits = min(len(relevant), k)
    idcg = discounts[:ideal_hits].sum()

    ndcg = dcg / idcg if idcg else 0

    return {
        f"Recall@{k}": recall,
        f"MRR@{k}": mrr,
        f"NDCG@{k}": ndcg,
        f"Hit@{k}": hit,
        f"Precision@{k}": precision,
    }

## Learning the Hybrid Weights

The regression model uses the normalized scores produced by ItemKNN and
SLIMElastic as its two input features.

For each observed validation interaction, the corresponding item is assigned a
target value of 1. Ten unobserved items are sampled as target-0 examples.
Items already present in the user's training history and validation positives
are excluded from negative sampling.

This produces training examples of the form:

$$
X = [s_{ItemKNN}, s_{SLIM}], \qquad y \in \{0,1\}
$$

The model coefficients are then used as the weights of the hybrid recommender.

In [46]:
def make_regression_data(eval_data, n_negatives=10):
    X = []
    y = []

    rng = np.random.default_rng(SEED)

    with torch.no_grad():

        for (
            interaction,
            history_index,
            positive_u,
            positive_i
        ) in eval_data:

            interaction = interaction.to(
                item_config["device"]
            )

            item_scores, slim_scores = get_scores(
                interaction
            )

            # Relevant items per user
            positives = {}

            for user, item in zip(
                positive_u.tolist(),
                positive_i.tolist()
            ):
                positives.setdefault(
                    user, set()
                ).add(item)

            # Training history per user
            history = {}

            if history_index is not None:
                history_u, history_i = history_index

                for user, item in zip(
                    history_u.tolist(),
                    history_i.tolist()
                ):
                    history.setdefault(
                        user, set()
                    ).add(item)

            n_items = item_scores.shape[1]

            for user, relevant in positives.items():

                # Positive examples
                for item in relevant:

                    X.append([
                        item_scores[user, item].item(),
                        slim_scores[user, item].item()
                    ])

                    y.append(1)

                # Possible negative items
                excluded = (
                    history.get(user, set())
                    | relevant
                    | {0}
                )

                candidates = [
                    item
                    for item in range(1, n_items)
                    if item not in excluded
                ]

                negatives = rng.choice(
                    candidates,
                    size=min(
                        n_negatives * len(relevant),
                        len(candidates)
                    ),
                    replace=False
                )

                for item in negatives:

                    X.append([
                        item_scores[user, item].item(),
                        slim_scores[user, item].item()
                    ])

                    y.append(0)

    return np.array(X), np.array(y)

In [47]:
X, y = make_regression_data(valid_data)

print("X:", X.shape)
print("y:", y.shape)

X: (105556, 2)
y: (105556,)


In [48]:
regression = LinearRegression()
regression.fit(X, y)

weights = regression.coef_

print("ItemKNN weight:", weights[0])
print("SLIM weight:", weights[1])
print("Intercept:", regression.intercept_)

ItemKNN weight: 0.1130620332476002
SLIM weight: 0.9413609228572593
Intercept:

 -0.0016842312174608493


## Learned Weights

Linear regression assigned a substantially larger coefficient to SLIMElastic
than to ItemKNN:

- ItemKNN: 0.1131
- SLIMElastic: 0.9414
- Intercept: -0.0017

This suggests that SLIMElastic provides most of the ranking signal, while
ItemKNN contributes a smaller complementary signal.

The intercept is not included when ranking items because it adds the same
constant to every candidate item's score and therefore does not affect their
ordering.

In [49]:
def evaluate_hybrid(eval_data, weights, k=10):
    results = []

    with torch.no_grad():

        for (
            interaction,
            history_index,
            positive_u,
            positive_i
        ) in eval_data:

            interaction = interaction.to(
                item_config["device"]
            )

            item_scores, slim_scores = get_scores(
                interaction
            )

            scores = (
                weights[0] * item_scores
                + weights[1] * slim_scores
            )

            # Padding item
            scores[:, 0] = -torch.inf

            # Already-seen items
            if history_index is not None:
                history_u, history_i = history_index

                scores[
                    history_u.to(scores.device),
                    history_i.to(scores.device)
                ] = -torch.inf

            top_items = torch.topk(
                scores,
                k=k,
                dim=1
            ).indices.cpu()

            positives = {}

            for user, item in zip(
                positive_u.tolist(),
                positive_i.tolist()
            ):
                positives.setdefault(
                    user, set()
                ).add(item)

            for user, relevant in positives.items():

                results.append(
                    metrics_at_k(
                        top_items[user].tolist(),
                        relevant,
                        k
                    )
                )

    return {
        metric: np.mean([
            result[metric]
            for result in results
        ])
        for metric in results[0]
    }

In [40]:
test_result = evaluate_hybrid(
    test_data,
    weights
)

pd.Series(test_result)

Recall@10       0.275670
MRR@10          0.523756
NDCG@10         0.323673
Hit@10          0.813362
Precision@10    0.217179
dtype: float64

In [50]:
sanity = pd.DataFrame({
    "ItemKNN": evaluate_hybrid(test_data, [1, 0]),
    "SLIM": evaluate_hybrid(test_data, [0, 1]),
    "Hybrid": evaluate_hybrid(test_data, weights),
}).T

sanity

,Recall@10,MRR@10,NDCG@10,Hit@10,Precision@10
ItemKNN,0.240213,0.482669,0.281920,0.792153,0.183033
SLIM,0.264639,0.511346,0.310771,0.803818,0.207105
Hybrid,0.265696,0.512514,0.312579,0.806999,0.208908


## Results

The regression-weighted hybrid achieved an MRR@10 of **0.5125**, compared with
**0.5113** for SLIMElastic and **0.4827** for ItemKNN under the same evaluation
procedure.

The hybrid also produced small improvements over SLIMElastic in Recall@10,
NDCG@10, Hit@10, and Precision@10. This indicates that although SLIMElastic
dominates the learned weighting, ItemKNN contributes complementary information
that slightly improves the final ranking.

Overall, the regression model successfully learned the contribution of each
individual recommender and produced a weighted hybrid that performed better
than either component under the same evaluation procedure.